In [2]:
%pip install -q sentence-transformers qdrant-client openai

In [3]:
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient, models
from openai import OpenAI
import getpass

In [5]:
data = [
    {
        "question": "What is RAG?",
        "answer": (
            "Retrieval-Augmented Generation (RAG) retrieves relevant "
            "information from a knowledge source and provides it to a "
            "language model to help generate an answer."
        )
    },
    {
        "question": "What are embeddings?",
        "answer": (
            "Embeddings are numerical vectors that represent the meaning "
            "of text and help computers find semantically similar content."
        )
    },
    {
        "question": "What is a vector database?",
        "answer": (
            "A vector database stores vector embeddings and supports "
            "similarity searches to retrieve relevant information."
        )
    },
    {
        "question": "What is a language model?",
        "answer": (
            "A language model is an AI model trained to understand "
            "and generate human language."
        )
    },
    {
        "question": "Why is RAG useful?",
        "answer": (
            "RAG provides a language model with relevant external "
            "information, helping it answer questions using a specific "
            "knowledge source."
        )
    }
]

print("Number of records:", len(data))

Number of records: 5


In [6]:
documents = [
    {
        "text": f"Question: {item['question']} Answer: {item['answer']}",
        "question": item["question"],
        "answer": item["answer"]
    }
    for item in data
]

print(documents[0]["text"])

Question: What is RAG? Answer: Retrieval-Augmented Generation (RAG) retrieves relevant information from a knowledge source and provides it to a language model to help generate an answer.


In [7]:
encoder = SentenceTransformer("all-MiniLM-L6-v2")

texts = [doc["text"] for doc in documents]

embeddings = encoder.encode(
    texts,
    convert_to_numpy=True
)

print("Number of embeddings:", len(embeddings))
print("Vector dimensions:", embeddings.shape[1])

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Number of embeddings: 5
Vector dimensions: 384


In [8]:
qdrant = QdrantClient(":memory:")

collection_name = "my_rag_knowledge"

qdrant.create_collection(
    collection_name=collection_name,
    vectors_config=models.VectorParams(
        size=embeddings.shape[1],
        distance=models.Distance.COSINE
    )
)

points = [
    models.PointStruct(
        id=idx,
        vector=embeddings[idx].tolist(),
        payload=documents[idx]
    )
    for idx in range(len(documents))
]

qdrant.upload_points(
    collection_name=collection_name,
    points=points,
    wait=True
)

print("Knowledge base created successfully!")

Knowledge base created successfully!


In [9]:
def retrieve_documents(question, limit=3):
    query_embedding = encoder.encode(question).tolist()

    results = qdrant.query_points(
        collection_name=collection_name,
        query=query_embedding,
        limit=limit,
        with_payload=True
    ).points

    return [
        {
            "text": result.payload["text"],
            "score": result.score
        }
        for result in results
    ]


question = "Why do we need RAG?"

search_results = retrieve_documents(question)

for result in search_results:
    print("Score:", round(result["score"], 3))
    print(result["text"])
    print("-" * 50)

Score: 0.54
Question: Why is RAG useful? Answer: RAG provides a language model with relevant external information, helping it answer questions using a specific knowledge source.
--------------------------------------------------
Score: 0.518
Question: What is RAG? Answer: Retrieval-Augmented Generation (RAG) retrieves relevant information from a knowledge source and provides it to a language model to help generate an answer.
--------------------------------------------------
Score: -0.011
Question: What are embeddings? Answer: Embeddings are numerical vectors that represent the meaning of text and help computers find semantically similar content.
--------------------------------------------------


In [10]:

from openai import OpenAI
import getpass

api_key = getpass.getpass("Enter your Gemini API key: ")

client = OpenAI(
    api_key=api_key,
    base_url="https://generativelanguage.googleapis.com/v1beta/"
)

print("API client is ready!")

Enter your Gemini API key: ··········
API client is ready!


In [12]:
def answer_question(question):
    # 1. Retrieve relevant documents
    results = retrieve_documents(question, limit=3)

    # 2. Prepare context for the language model
    context = "\n\n".join(
        result["text"] for result in results
    )

    # 3. Generate an answer using the retrieved context
    completion = client.chat.completions.create(
        model="gemini-3.8-flash",
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a helpful AI assistant. "
                    "Answer using only the provided context. "
                    "If the context does not contain the answer, "
                    "say that the information is not available."
                )
            },
            {
                "role": "user",
                "content": (
                    f"Context:\n{context}\n\n"
                    f"Question: {question}"
                )
            }
        ],
        temperature=0
    )

    return completion.choices[0].message.content


question = "Why do we need RAG?"
answer = answer_question(question)

print("Question:", question)
print("Answer:", answer)

Question: Why do we need RAG?
Answer: Based on the provided context, RAG is useful/needed because it provides a language model with relevant external information, helping it answer questions using a specific knowledge source.


In [16]:
test_questions = [
    "What are embeddings?",
    "What is a vector database?",
    "What is a language model?",
    "What is RAG?"
]

for question in test_questions:
    print("Question:", question)
    print("Answer:", answer_question(question))
    print("-" * 60)

Question: What are embeddings?
Answer: Embeddings are numerical vectors that represent the meaning of text and help computers find semantically similar content.
------------------------------------------------------------
Question: What is a vector database?
Answer: A vector database stores vector embeddings and supports similarity searches to retrieve relevant information.
------------------------------------------------------------
Question: What is a language model?
Answer: A language model is an AI model trained to understand and generate human language.
------------------------------------------------------------
Question: What is RAG?
Answer: Retrieval-Augmented Generation (RAG) retrieves relevant information from a knowledge source and provides it to a language model to help generate an answer.
------------------------------------------------------------


In [17]:
%pip install -q datasets

In [18]:
print("Dataset library is ready!")

Dataset library is ready!


In [29]:


from datasets import load_dataset

dataset = load_dataset(
    "prem7030/General-Knowledge",
    split="train[:100]"
)

print("Number of records:", len(dataset))
print("\nAvailable columns:", dataset.column_names)
print("\nFirst record:")
print(dataset[0])

README.md:   0%|          | 0.00/1.78k [00:00<?, ?B/s]

output.json: reconstructing file:   0%|          |  0.00B / 16.2MB            

output.json: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/37635 [00:00<?, ? examples/s]

Number of records: 100

Available columns: ['Question', 'Answer']

First record:
{'Question': 'What is Artificial Intelligence?', 'Answer': 'Artificial Intelligence refers to the development of computer systems that can perform tasks that would typically require human intelligence, such as visual perception, speech recognition, decision-making, and language translation.\\n'}


In [30]:


documents = []

for item in dataset:
    question = item["Question"].strip()
    answer = item["Answer"].strip()

    document_text = (
        f"Question: {question}\n"
        f"Answer: {answer}"
    )

    documents.append({
        "text": document_text,
        "question": question,
        "answer": answer
    })

print("Number of documents:", len(documents))
print("\nFirst document:")
print(documents[0])

Number of documents: 100

First document:
{'text': 'Question: What is Artificial Intelligence?\nAnswer: Artificial Intelligence refers to the development of computer systems that can perform tasks that would typically require human intelligence, such as visual perception, speech recognition, decision-making, and language translation.\\n', 'question': 'What is Artificial Intelligence?', 'answer': 'Artificial Intelligence refers to the development of computer systems that can perform tasks that would typically require human intelligence, such as visual perception, speech recognition, decision-making, and language translation.\\n'}


In [51]:


ai_keywords = [
    "artificial intelligence",
    "machine learning",
    "deep learning",
    "neural network",
    "AI algorithm",
    "supervised learning",
    "unsupervised learning",
    "reinforcement learning",
    "transfer learning",
    "overfitting",
    "underfitting",
    "explainable AI",
    "chatbot",
    "recommendation system",
    "Turing Test",
    "natural language processing",
    "natural language generation",
    "computer vision",
    "sentiment analysis",
    "generative AI",
    "large language model",
    "convolutional neural network",
    "AGI",
    "robotics"
]

ai_documents = []
seen_questions = set()

for doc in documents:
    question = doc["question"].lower()

    if any(keyword.lower() in question for keyword in ai_keywords):
        normalized_question = question.strip()

        if normalized_question not in seen_questions:
            ai_documents.append(doc)
            seen_questions.add(normalized_question)

print("Original documents:", len(documents))
print("AI documents:", len(ai_documents))
print("\nFirst AI document:")
print(ai_documents[0])

Original documents: 100
AI documents: 26

First AI document:
{'text': 'Question: What is Artificial Intelligence?\nAnswer: Artificial Intelligence refers to the development of computer systems that can perform tasks that would typically require human intelligence, such as visual perception, speech recognition, decision-making, and language translation.\\n', 'question': 'What is Artificial Intelligence?', 'answer': 'Artificial Intelligence refers to the development of computer systems that can perform tasks that would typically require human intelligence, such as visual perception, speech recognition, decision-making, and language translation.\\n'}


In [52]:



texts = [doc["text"] for doc in ai_documents]

embeddings = encoder.encode(
    texts,
    convert_to_numpy=True,
    show_progress_bar=True
)

print("Number of embeddings:", len(embeddings))
print("Vector dimensions:", embeddings.shape[1])

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Number of embeddings: 26
Vector dimensions: 384


In [53]:



from qdrant_client import models

collection_name = "ai_knowledge_base"

qdrant.recreate_collection(
    collection_name=collection_name,
    vectors_config=models.VectorParams(
        size=embeddings.shape[1],
        distance=models.Distance.COSINE
    )
)

points = [
    models.PointStruct(
        id=idx,
        vector=embeddings[idx].tolist(),
        payload=ai_documents[idx]
    )
    for idx in range(len(ai_documents))
]

qdrant.upload_points(
    collection_name=collection_name,
    points=points,
    wait=True
)

print("AI knowledge base updated successfully!")
print("Documents stored:", len(points))

AI knowledge base updated successfully!
Documents stored: 26


/tmp/ipykernel_1552/907654227.py:5: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  qdrant.recreate_collection(


In [70]:
question = "How does an AI agent learn to choose actions based on rewards?"
search_results = retrieve_documents(question)

for i, result in enumerate(search_results, start=1):
    print(f"\nResult {i}")
    print("Similarity score:", round(result["score"], 3))
    print("Question:", result["question"])
    print("Answer:", result["answer"])
    print("-" * 50)


Result 1
Similarity score: 0.547
Question: What is Reinforcement Learning?
Answer: Reinforcement Learning is a subset of Machine Learning where an algorithm learns to take actions in an environment to maximize a reward signal. The algorithm receives feedback in the form of a reward or penalty based on the actions it takes, and uses this feedback to improve its decision-making over time.\n
--------------------------------------------------

Result 2
Similarity score: 0.375
Question: What is an AI algorithm?
Answer: An AI algorithm is a set of rules or instructions that a computer program follows to make decisions or perform tasks in an intelligent way.\n
--------------------------------------------------

Result 3
Similarity score: 0.351
Question: What is the difference between AI and machine learning?
Answer: AI refers to the simulation of human intelligence in machines, while machine learning is a subset of AI that allows machines to learn and improve from experience without being ex

In [35]:

def retrieve_documents(question, limit=3):
    query_embedding = encoder.encode(question).tolist()

    results = qdrant.query_points(
        collection_name="ai_knowledge_base",
        query=query_embedding,
        limit=limit,
        with_payload=True
    ).points

    return [
        {
            "text": result.payload["text"],
            "question": result.payload["question"],
            "answer": result.payload["answer"],
            "score": result.score
        }
        for result in results
    ]


question = "What is Artificial Intelligence?"
search_results = retrieve_documents(question)

for i, result in enumerate(search_results, start=1):
    print(f"\nResult {i}")
    print("Similarity score:", round(result["score"], 3))
    print("Stored question:", result["question"])
    print("Expected answer:", result["answer"])
    print("-" * 50)


Result 1
Similarity score: 0.872
Stored question: What is Artificial Intelligence?
Expected answer: Artificial Intelligence refers to the development of computer systems that can perform tasks that would typically require human intelligence, such as visual perception, speech recognition, decision-making, and language translation.\n
--------------------------------------------------

Result 2
Similarity score: 0.683
Stored question: What is an AI algorithm?
Expected answer: An AI algorithm is a set of rules or instructions that a computer program follows to make decisions or perform tasks in an intelligent way.\n
--------------------------------------------------

Result 3
Similarity score: 0.648
Stored question: What is artificial general intelligence (AGI)?
Expected answer: Artificial general intelligence (AGI) refers to the hypothetical ability of machines to understand or learn any intellectual task that a human being can do.\n
--------------------------------------------------


In [68]:


def answer_question(question):
    results = retrieve_documents(question, limit=3)

    context = "\n\n".join(
        result["text"] for result in results
    )

    completion = client.chat.completions.create(
        model="gemini-3.8-flash",
        messages=[
            {
                "role": "system",
                "content": (
                    "You are an AI knowledge assistant. "
                    "Answer the user's question using the provided context. "
                    "The question may be phrased differently from the stored questions. "
                    "Use relevant information from the context to answer in your own words. "
                    "If the context is insufficient to answer, clearly say what is missing. "
                    "Do not invent facts."
                )
            },
            {
                "role": "user",
                "content": (
                    f"Context:\n{context}\n\n"
                    f"Question: {question}"
                )
            }
        ],
        temperature=0
    )

    return completion.choices[0].message.content


question = "What is Reinforcement Learning?"
answer = answer_question(question)

print("Question:", question)
print("Generated answer:", answer)

RateLimitError: Error code: 429 - [{'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.8-flash\nPlease retry in 14h38m27.157511075s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-3.8-flas